# Jev for Sales Lead Qualification & Routing

In this notebook, we will build a simple AI-powered lead qualification
and routing system using Jev.

Imagine a company receiving hundreds or thousands of leads from:

- Website forms
- Contact-us pages
- Email
- LinkedIn
- Product demos
- Chatbots

Every lead needs to be evaluated.

Our system will use Jev to answer questions such as:

- Is this a genuine business lead?
- What type of customer is this?
- How strong is the buying intent?
- Should a salesperson follow up?
- Which sales team should receive the lead?

By the end of this notebook, we will have a reusable lead
qualification pipeline.

##  What are we building?

Consider this incoming lead:

> "We are a 500-person company currently using Snowflake.
> We are evaluating Databricks for our data engineering team
> and would like to schedule a demo next week."

A human salesperson can immediately understand that this
is potentially valuable.

But imagine receiving 5,000 such messages.

We need a consistent way to classify them.


##  Why use Jev here?

Lead qualification mostly involves structured decisions.

For example:

    Is this a real business opportunity?
    YES / NO

    What type of customer is this?
    Enterprise / SMB / Startup / Individual

    What is the buying intent?
    Low / Medium / High

These are exactly the kinds of decisions that can be represented
as structured Jev questions.

Instead of asking an LLM to generate a long explanation,
we can ask Jev for the structured information our application needs.

In [2]:
import os 
from dotenv import load_dotenv
load_dotenv()
from langchain_typesafe import (
    TypeSafeClassifier,
    Noul,
    Choice,
    Score
)

classifier = TypeSafeClassifier()

C:\Users\sarvesh\AppData\Local\Temp\ipykernel_21244\1016552830.py:11: LangChainBetaWarning: The class `TypeSafeClassifier` is in beta. It is actively being worked on, so the API may change.
  classifier = TypeSafeClassifier()


##  Start with a simple lead

Let's start with one realistic lead.

Our first question:

> Is this a genuine business sales lead?

This is a YES/NO decision, so we can use `Noul`.

In [3]:
lead = """
We are a 500-person company currently using Snowflake.
We are evaluating Databricks for our data engineering team
and would like to schedule a product demo next week.
"""

response = classifier.invoke({
    "state": lead,
    
    "questions": {
        "is_qualified": Noul(
            instructions="""
            Is this a genuine business sales lead with
            a potential commercial opportunity?
            """
        )
    }
})

qualification_probability = response.nouls["is_qualified"].noul

print("Qualification probability:", qualification_probability)

Qualification probability: 0.85


## Understanding the result

Suppose Jev returns:

    0.97

This means Jev estimates a high probability that the answer
to our YES/NO question is YES.

We can now define application logic.

For example:

    probability > 0.80
        → treat as qualified

    probability <= 0.80
        → send for further review

The threshold is an application-level decision and should be
validated using real business data.

## Identifying the type of lead

Qualification alone isn't enough.

Different sales teams may handle different customer segments.

For example:

- Enterprise
- SMB
- Startup
- Individual

We can use a `Choice` question to classify the lead.

In [4]:
response = classifier.invoke({
    "state": lead,
    
    "questions": {
        "lead_type": Choice(
            instructions="What type of customer is this lead?",
            
            criteria={
                "enterprise": """
                Large organization with significant employees,
                teams, budget, or enterprise-scale requirements.
                """,
                
                "smb": """
                Small or medium-sized business with relatively
                smaller teams or requirements.
                """,
                
                "startup": """
                Early-stage company or startup building or
                scaling its product/business.
                """,
                
                "individual": """
                Individual person evaluating the product for
                personal or non-business use.
                """
            }
        )
    }
})

lead_type = response.choices["lead_type"].choice

print("Lead type:", lead_type)

Lead type: enterprise


##  Understanding buying intent

Not every qualified lead is ready to buy.

Compare:

### Lead A

> "Can you send me information about your product?"

versus:

### Lead B

> "We're evaluating three vendors and would like to
> schedule a demo next week."

Both may be legitimate leads.

But their buying intent is different.

Let's classify intent into:

- Low
- Medium
- High

In [5]:
response = classifier.invoke({
    "state": lead,
    
    "questions": {
        "buying_intent": Score(
            instructions="How strong is the customer's current buying intent?",
            
            criteria=[
                "low",
                "medium",
                "high"
            ]
        )
    }
})

intent = response.scores["buying_intent"].score

print("Buying intent:", intent)

Buying intent: 1.93


##  Evaluate a lead using multiple questions

Our complete lead evaluation will ask Jev:

1. Is this a qualified business lead?
2. What type of customer is it?
3. How strong is the buying intent?
4. Does the lead explicitly request a sales interaction?

All of these decisions are based on the same lead information.

In [6]:
response = classifier.invoke({
    "state": lead,
    
    "questions": {
        
        "is_qualified": Noul(
            instructions="""
            Is this a genuine business sales lead with
            a potential commercial opportunity?
            """
        ),
        
        "lead_type": Choice(
            instructions="What type of customer is this lead?",
            
            criteria={
                "enterprise": """
                Large organization with significant employees,
                teams, budget, or enterprise-scale requirements.
                """,
                
                "smb": """
                Small or medium-sized business.
                """,
                
                "startup": """
                Early-stage company or startup.
                """,
                
                "individual": """
                Individual person without a clear business use case.
                """
            }
        ),
        
        "buying_intent": Score(
            instructions="How strong is the customer's current buying intent?",
            
            criteria=[
                "low",
                "medium",
                "high"
            ]
        ),
        
        "requests_sales_contact": Noul(
            instructions="""
            Does the lead explicitly request a demo, sales call,
            meeting, pricing discussion, proposal, or similar
            sales interaction?
            """
        )
    }
})

print("Qualified:")
print(response.nouls["is_qualified"].noul)

print("\nLead type:")
print(response.choices["lead_type"].choice)

print("\nBuying intent:")
print(response.scores["buying_intent"].score)

print("\nRequests sales contact:")
print(response.nouls["requests_sales_contact"].noul)

Qualified:
0.85

Lead type:
enterprise

Buying intent:
1.92

Requests sales contact:
0.98


#### Preparing reusable function

In [7]:
def evaluate_lead(lead):
    
    response = classifier.invoke({
        "state": lead,
        
        "questions": {
            
            "is_qualified": Noul(
                instructions="""
                Is this a genuine business sales lead with
                a potential commercial opportunity?
                """
            ),
            
            "lead_type": Choice(
                instructions="What type of customer is this lead?",
                
                criteria={
                    "enterprise": """
                    Large organization with significant employees,
                    teams, budget, or enterprise-scale requirements.
                    """,
                    
                    "smb": """
                    Small or medium-sized business.
                    """,
                    
                    "startup": """
                    Early-stage company or startup.
                    """,
                    
                    "individual": """
                    Individual person without a clear business use case.
                    """
                }
            ),
            
            "buying_intent": Score(
                instructions="How strong is the customer's current buying intent?",
                
                criteria=[
                    "low",
                    "medium",
                    "high"
                ]
            ),
            
            "requests_sales_contact": Noul(
                instructions="""
                Does the lead explicitly request a demo, sales call,
                meeting, pricing discussion, proposal, or similar
                sales interaction?
                """
            )
        }
    })
    
    return {
        "qualification_probability":
            response.nouls["is_qualified"].noul,
        
        "lead_type":
            response.choices["lead_type"].choice,
        
        "buying_intent":
            response.scores["buying_intent"].score,
        
        "requests_sales_contact":
            response.nouls["requests_sales_contact"].noul
    }


result = evaluate_lead(lead)

result

{'qualification_probability': 0.86,
 'lead_type': 'enterprise',
 'buying_intent': 1.89,
 'requests_sales_contact': 0.98}

#### Building a router

In [9]:
def route_lead(result):
    
    qualification = result["qualification_probability"]
    lead_type = result["lead_type"]
    intent = result["buying_intent"]
    requested_contact = result["requests_sales_contact"]
    
    # First check whether this looks like a qualified lead
    if qualification < 0.70:
        return "Marketing / Nurturing"
    
    # High-intent leads requesting sales interaction
    if intent == "high" and requested_contact > 0.70:
        
        if lead_type == "enterprise":
            return "Enterprise Sales"
        
        elif lead_type == "smb":
            return "SMB Sales"
        
        elif lead_type == "startup":
            return "Startup / Growth Sales"
    
    # Other qualified leads
    return "Sales Development / Lead Nurturing"

routing = route_lead(result)

print("Recommended routing:", routing)

Recommended routing: Sales Development / Lead Nurturing


This notebook demonstrates a technical pattern,
not a production-ready lead scoring system.